# 02 — Phase 1: GQA Only Baseline

**配置**: Qwen2.5-1.5B-Instruct + HuggingFace DynamicCache (FP16)

这是最基本的 baseline：
- **GQA**: 12 query heads, 2 KV heads → KV Cache 比 MHA 小 6×
- **DynamicCache**: 连续 FP16 张量，逐步 concat 扩展
- 没有量化，没有分页

### 关键目标
1. 记录各指标基线值 (TTFT, TPOT, peak memory)
2. **找到 OOM 阈值**：context 多长会炸掉 10-12 GB 统一内存
3. 分离 **模型权重占用** vs **KV Cache 占用**

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.dataset_utils import load_prompts

In [ ]:

from src.jetson_utils import load_model_safe, aggressive_cleanup

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FALLBACK_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"

# Safe loading: checks free memory, falls back to lighter model, uses low_cpu_mem_usage
model, tokenizer = load_model_safe(
    MODEL_NAME,
    fallback_name=FALLBACK_NAME,
    device=DEVICE,
)

# 分离显示模型权重 vs 可用空间
budget = print_memory_budget(model, DEVICE)


In [ ]:
# GQA 架构确认
cfg = model.config
print(f"num_attention_heads  : {cfg.num_attention_heads}")
print(f"num_key_value_heads  : {cfg.num_key_value_heads}")
print(f"num_hidden_layers    : {cfg.num_hidden_layers}")
print(f"hidden_size          : {cfg.hidden_size}")
head_dim = cfg.hidden_size // cfg.num_attention_heads
print(f"head_dim             : {head_dim}")
print(f"GQA ratio            : {cfg.num_attention_heads // cfg.num_key_value_heads}:1")
print(f"\n→ 每个 token 的 KV Cache (FP16):")
kv_per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * 2
print(f"  {kv_per_token:,} bytes = {kv_per_token/1024:.1f} KB/token")

---
## OOM 阈值探测

在 10-12 GB 统一内存下，找出 GQA-only baseline 能支撑的最大 context length。

In [ ]:
# OOM 阈值探测
print("探测 OOM 阈值 (context length → peak memory)...")
oom_result = find_oom_threshold(
    model, tokenizer,
    context_lengths=[256, 512, 1024, 2048, 4096, 8192, 16384],
    max_new_tokens=32,
    cache_factory=None,  # default DynamicCache
)

print(f"\n最大安全长度: {oom_result['max_safe_length']}")
print(f"OOM 发生在  : {oom_result['oom_length']}")
print(f"\n详细结果:")
for r in oom_result['results']:
    status = r['status']
    if status == 'ok':
        print(f"  ctx={r['context_length']:>6}  peak={r['peak_memory_mb']:>7.0f} MB  "
              f"util={r['utilization']*100:>5.1f}%")
    else:
        print(f"  ctx={r['context_length']:>6}  → {status}")

In [ ]:
# OOM 阈值可视化
import matplotlib.pyplot as plt

ok_results = [r for r in oom_result['results'] if r['status'] == 'ok']
ctx_lens = [r['context_length'] for r in ok_results]
peak_mems = [r['peak_memory_mb'] for r in ok_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ctx_lens, peak_mems, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axhline(y=JETSON_USABLE_GB * 1024, color='red', linestyle='--',
           label=f'Jetson budget ({JETSON_USABLE_GB} GB)')
if oom_result['oom_length']:
    ax.axvline(x=oom_result['oom_length'], color='darkred', linestyle=':',
               label=f'OOM @ {oom_result["oom_length"]}')
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Peak Memory (MB)')
ax.set_title('GQA-Only: Memory vs Context Length')
ax.legend()
ax.set_xscale('log', base=2)
plt.tight_layout()
plt.savefig('../results/oom_threshold_gqa.png', dpi=150)
plt.show()

---
## PubMedQA Benchmark

In [ ]:
prompts = load_prompts('../results/pubmedqa_prompts.json')
print(f"Loaded {len(prompts)} prompts")

In [ ]:
# 单样本快速测试
m = measure_generation(model, tokenizer, prompts[0]['prompt'], max_new_tokens=64)
print(f"TTFT           : {m.ttft_ms:.1f} ms")
print(f"TPOT           : {m.tpot_ms:.1f} ms")
print(f"Model weights  : {m.model_weight_mb:.0f} MB")
print(f"KV Cache       : {m.kv_cache_memory_mb:.1f} MB")
print(f"Peak memory    : {m.peak_memory_mb:.0f} MB")
print(f"Memory util    : {m.memory_utilization*100:.1f}% of {JETSON_USABLE_GB}GB budget")
print(f"Fragmentation  : {m.memory_fragmentation:.3f}")
print(f"Tokens         : {m.num_input_tokens} in → {m.num_output_tokens} out")
print(f"\nGenerated: {m.generated_text[:200]}...")

In [ ]:
# 完整基准测试
results_gqa = run_benchmark(
    model, tokenizer, prompts,
    cache_factory=None,     # HF DynamicCache
    max_new_tokens=256,
    warmup_runs=2, num_runs=3,
)
print(f"Completed {len(results_gqa)} samples")

In [ ]:
import pandas as pd

df = pd.DataFrame(results_gqa)
df['config'] = 'GQA_only'
df.to_csv('../results/baseline_gqa.csv', index=False)

cols = ['ttft_ms', 'tpot_ms', 'peak_memory_mb', 'model_weight_mb',
        'kv_cache_memory_mb', 'memory_utilization']
print(df[cols].describe().round(1))

---
## PPL 评估（质量基线）

记录全精度基线的 Perplexity，后续对比量化后的退化。

In [ ]:
from src.perplexity import compute_perplexity

# 用 PubMedQA 的 long_answer 作为评估文本
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]
print(f"PPL evaluation on {len(eval_texts)} texts...")

ppl_baseline = compute_perplexity(model, tokenizer, eval_texts, max_length=512)
print(f"\nBaseline PPL : {ppl_baseline['ppl']:.2f}")
print(f"Avg loss     : {ppl_baseline['avg_loss']:.4f}")
print(f"Tokens eval  : {ppl_baseline['num_tokens']}")

In [ ]:
# 保存 PPL 结果
import json
ppl_results = {'GQA_only': ppl_baseline}
with open('../results/ppl_results.json', 'w') as f:
    json.dump(ppl_results, f, indent=2)
print("Saved → results/ppl_results.json")

In [ ]:
# 释放模型（后续 notebook 重新加载）
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")